# Exploratory Data Analysis (EDA) - Energy Demand Network
En este notebook realizaremos el Análisis Exploratorio de Datos (EDA) sobre los conjuntos descargados desde ENTSO-E (Europa) y ESIOS (Cataluña). El objetivo es entender la distribución, calidad, estacionalidad y correlación temporal/espacial de las variables antes de entrenar nuestra Graph Neural Network (GNN).

## 0. Preparación y Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 6)

## 1. Carga de Datos
Cargamos todos los CSVs separados por las APIs y los unimos utilizando el índice temporal (fechas).

In [ ]:
def load_and_merge_data():
    # Cargar datos de Europa (ENTSO-E)
    europe_files = glob.glob("../data/raw/europe/*.csv")
    df_europe = pd.DataFrame()
    for f in europe_files:
        col_name = os.path.basename(f).replace(".csv", "")
        tmp = pd.read_csv(f, parse_dates=["utc_timestamp"])
        # Clean timezone to pure UTC for merging safely
        tmp["utc_timestamp"] = pd.to_datetime(tmp["utc_timestamp"], utc=True)
        tmp.set_index("utc_timestamp", inplace=True)
        
        if df_europe.empty:
            df_europe = tmp
        else:
            # Some files might have duplicated indices due to DST changes
            tmp = tmp[~tmp.index.duplicated(keep="first")]
            df_europe = df_europe.join(tmp, how="outer")
            
    # Cargar datos de Cataluña (ESIOS)
    cat_files = glob.glob("../data/raw/catalonia/*.csv")
    df_cat = pd.DataFrame()
    for f in cat_files:
        col_name = os.path.basename(f).replace(".csv", "")
        tmp = pd.read_csv(f, parse_dates=["datetime"])
        tmp["datetime"] = pd.to_datetime(tmp["datetime"], utc=True)
        tmp.set_index("datetime", inplace=True)
        # Rename ESIOS default feature columns to specific parameter names
        if len(tmp.columns) == 1:
            tmp.columns = [col_name]
        
        if df_cat.empty:
            df_cat = tmp
        else:
            tmp = tmp[~tmp.index.duplicated(keep="first")]
            df_cat = df_cat.join(tmp, how="outer")
            
    # Unir ambos datasets
    df_full = df_europe.join(df_cat, how="outer")
    return df_europe, df_cat, df_full

df_eu, df_cat, df = load_and_merge_data()
print(f"Dataset Global Formato: {df.shape}")
df.head()

## 2. Análisis de Valores Nulos (Quality Check)

In [ ]:
print("Valores Nulos por columna:\n")
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
print(pd.DataFrame({"Nulos": missing, "Porcentaje (%)": missing_pct}).sort_values(by="Porcentaje (%)", ascending=False))

## 3. Serie Temporal Principal: Demanda en Cataluña
Visualizamos nuestra variable objetivo (Target).

In [ ]:
if "esios_demand_real" in df.columns:
    df["esios_demand_real"].resample("W").mean().plot(color="orange")
    plt.title("Evolución de la Demanda Real en Cataluña (Media Semanal)")
    plt.ylabel("Demanda (MW)")
    plt.show()
else:
    print("Variable de demanda real catalana no encontrada, comprueba la ejecución del script downloader.")

Exploramos la de estacionalidad (día vs noche, semana vs fin de semana, invierno vs verano).

In [ ]:
if "esios_demand_real" in df.columns:
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 2, 1)
    sns.boxplot(x=df.index.hour, y=df["esios_demand_real"])
    plt.title("Estacionalidad Diaria (Hora)")
    
    plt.subplot(1, 2, 2)
    sns.boxplot(x=df.index.dayofweek, y=df["esios_demand_real"])
    plt.title("Estacionalidad Semanal (Día de la semana)")
    plt.xticks(ticks=range(7), labels=["L", "M", "X", "J", "V", "S", "D"])
    plt.show()

## 4. Análisis Espacial: Demanda Europea
Comparamos las escalas y distribuciones de los distintos nodos (países) del grafo.

In [ ]:
eu_demand_cols = [c for c in df.columns if "entsoe_demand" in c or len(c) == 2]
if eu_demand_cols:
    df[eu_demand_cols].resample("M").mean().plot(alpha=0.8)
    plt.title("Demanda Mensual Media en Países Europeos")
    plt.ylabel("MW")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.show()

## 5. Correlación Espacial y Variables Exógenas
Calculamos la matriz de correlación de Pearson para ver cómo los países y las variables afectan a nuestra variable objetivo (Cataluña).

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False)
plt.title("Matriz de Correlación Global")
plt.show()

In [ ]:
if "esios_demand_real" in corr_matrix.columns:
    target_corr = corr_matrix["esios_demand_real"].sort_values(ascending=False)
    print("Correlación con la Demanda Real de Cataluña:\n")
    print(target_corr)